# 3. Sharpness-aware minimization (SAM) wirh Sparse Networks - 25 points

## 3.1 Get a sparse networks through pruning
**Pruning** is a technique used to reduce the size and complexity of a neural network model by removing (setting to zero) less important parameters. The goal is to create a more efficient model that retains its predictive accuracy while being smaller, which can improve both inference speed and memory usage.

Let's train a simple model on the MNIST dataset to learn about pruning at first. We just use 10% of the dataset for both training and testing.

### 3.1.1 Train a dense network with SGD
Let us first train a dense model with SGD. We reuse the model for the discriminator of the GAN in Homework 2 and name it 'Classifier'.

In [39]:
from lib.part3.utils import *
max_epochs = 10
device = "cpu" # Change this if you can and want to use a GPU device
model = Classifier().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

We define the optimizing process of SGD.

In [40]:
def optimize_sgd(model, optimizer, img, label):
    optimizer.zero_grad()
    output = model(img)
    loss = cross_entropy(output, label)
    loss.backward()
    optimizer.step()

The following cell runs the training loop, this might take a few minutes.

In [41]:
train_model(model, optimizer, optimize_sgd, max_epochs)

Epoch 0 with 0.933 accuracy on the validation set.
Epoch 1 with 0.947 accuracy on the validation set.
Epoch 2 with 0.956 accuracy on the validation set.
Epoch 3 with 0.957 accuracy on the validation set.
Epoch 4 with 0.96 accuracy on the validation set.
Epoch 5 with 0.962 accuracy on the validation set.
Epoch 6 with 0.966 accuracy on the validation set.
Epoch 7 with 0.967 accuracy on the validation set.
Epoch 8 with 0.969 accuracy on the validation set.
Epoch 9 with 0.971 accuracy on the validation set.


#### Evaluate model
Evaluate the model on the test set.

In [42]:
acc = evaluate(model)
print(f"Accuracy of {round(acc, 4)} on the test set.")

Accuracy of 0.9771 on the test set.


### 3.1.2 Sparse network with magnitude-based pruning

Magnitude-based pruning specifically focuses on **removing weights that have the smallest absolute values**, under the assumption that weights with smaller magnitudes contribute less to the model's output.

**(1)** (6 points) Realize magnitude-based pruning below, which removes a part of weights that have the smallest absolute values.

In [43]:
def magnitude_prune(model, prune_fraction):
    for name, param in model.named_parameters():
        if "weight" in name and param.requires_grad:
            # FILL: Get weight's absolute values
            weight_abs = param.abs()
            # FILL: Compute the threshold
            threshold = weight_abs.view(-1).kthvalue(int(prune_fraction * weight_abs.numel())).values.item()
            # FILL: Prune weights below the threshold
            mask = weight_abs >= threshold
            param.data.mul_(mask)  # Apply the mask to set pruned weights to zero
    return model

In [44]:
import copy
# Copy a model for pruning
sparse_model = copy.deepcopy(model)
# Get a sparse model by pruning 50% parameters
sparse_model = magnitude_prune(sparse_model, prune_fraction=0.5)

Copy a sparse model for SAM implementation later in 3.2.

In [45]:
sparse_model_sam = copy.deepcopy(sparse_model)

Evaluation after pruning

In [46]:
acc = evaluate(sparse_model)
print(f"Accuracy of {round(acc, 4)} on the test set.")

Accuracy of 0.8738 on the test set.


### 3.1.3 Finetune the sparse model

We finetune the sparse model after pruning with SGD to recover its performance.

In [47]:
finetune_epoch = 10
train_model(sparse_model, optimizer, optimize_sgd, finetune_epoch)

Epoch 0 with 0.918 accuracy on the validation set.
Epoch 1 with 0.918 accuracy on the validation set.
Epoch 2 with 0.918 accuracy on the validation set.
Epoch 3 with 0.918 accuracy on the validation set.
Epoch 4 with 0.918 accuracy on the validation set.
Epoch 5 with 0.918 accuracy on the validation set.
Epoch 6 with 0.918 accuracy on the validation set.
Epoch 7 with 0.918 accuracy on the validation set.
Epoch 8 with 0.918 accuracy on the validation set.
Epoch 9 with 0.918 accuracy on the validation set.


Evaluate the sparse model after finetuning.

In [48]:
acc = evaluate(sparse_model)
print(f"Accuracy of {round(acc, 4)} on the test set.")

Accuracy of 0.9287 on the test set.


**(2)** (2 point) What are the pros and cons of sparse networks?

**pros:** improve inference, memory usage, scalability, interpretability and reduce overfitting

**cons:** implementation complexity to benefit from sparse networks and accuracy trade-off

## 3.2 Train the sparse model with SAM

Sharpness-aware minimization (SAM) is a new optimization technique, which is satisfied with not just a low loss, instead it seeks a neighborhood with uniformly low loss. SAM is motivated by the link between the geometry of the loss landscape and generalization. It makes sense that a low loss within a uniformly low loss neighborhood will generalize better than a low loss within a region of higher variance.

To be specific, we consider a model with the weight vector of $\mathbf{w}$ and the training loss $L_S$. SAM aims to minimize the maximum loss within a small region which is usually a $\ell_2$ ball with $\rho$ radius. Note that $\rho$ is a small value close to $0$. Therefore, SAM can be formulated as a minimax optimization problem:
$$\min_{\mathbf{w}} \max_{\mathbf{\epsilon}: \|\mathbf{\epsilon}\|_2\leq \rho} L_S (\mathbf{w} + \mathbf{\epsilon})$$

**(3)** (3 points) Please solve the inner maximum problem by first-order Taylor expansion.

The Taylor expansion of $ L_S(\mathbf{w} + \mathbf{\epsilon}) $ around $\mathbf{w}$ is given by:
$
L_S(\mathbf{w} + \mathbf{\epsilon}) \approx L_S(\mathbf{w}) + \nabla L_S(\mathbf{w})^\top \mathbf{\epsilon}
$

The inner maximization problem becomes:
$
\max_{\mathbf{\epsilon} : \|\mathbf{\epsilon}\|_2 \leq \rho} L_S(\mathbf{w} + \mathbf{\epsilon}) 
\approx \max_{\mathbf{\epsilon} : \|\mathbf{\epsilon}\|_2 \leq \rho} \big( L_S(\mathbf{w}) + \nabla L_S(\mathbf{w})^\top \mathbf{\epsilon} \big)
$

Since $ L_S(\mathbf{w}) $ is constant with respect to $\mathbf{\epsilon}$, we focus on maximizing:
$
\max_{\mathbf{\epsilon} : \|\mathbf{\epsilon}\|_2 \leq \rho} \nabla L_S(\mathbf{w})^\top \mathbf{\epsilon}
$

The solution to this maximization problem is to align $\mathbf{\epsilon}$ with the direction of $ \nabla L_S(\mathbf{w}) $, subject to the $ \ell_2 $-norm constraint $ \|\mathbf{\epsilon}\|_2 \leq \rho $. This gives:
$
\mathbf{\epsilon} = \rho \frac{\nabla L_S(\mathbf{w})}{\|\nabla L_S(\mathbf{w})\|_2}
$

Substituting $\mathbf{\epsilon}$ back into the problem, the inner maximum problem evaluates to:
$
\max_{\mathbf{\epsilon} : \|\mathbf{\epsilon}\|_2 \leq \rho} L_S(\mathbf{w} + \mathbf{\epsilon}) 
\approx L_S(\mathbf{w}) + \rho \|\nabla L_S(\mathbf{w})\|_2
$

- **Optimal perturbation**:
  $
  \mathbf{\epsilon} = \rho \frac{\nabla L_S(\mathbf{w})}{\|\nabla L_S(\mathbf{w})\|_2}
  $

- **Maximum value**:
  $
  L_S(\mathbf{w}) + \rho \|\nabla L_S(\mathbf{w})\|_2
  $

**(4)** (8 points) Now we will train the same model using the SAM optimizer.
Please implement SAM by the two steps below. The first step is for the maximizer which calculates $\epsilon$ obtained in question (1). The second step is the normal step for the minimizer: $\mathbf{w}_{t+1} = \mathbf{w}_{t} - \eta_t \nabla L_S (\mathbf{w}_t + \mathbf{\epsilon}_t)$ where $\eta_t$ is step size. Note that we set $\rho=0.05$.

Hint: be careful about weight updates.

In [49]:
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, lr=0.01, rho=0.05):
        assert rho >= 0.0, f"Invalid rho, should be non-negative: {rho}"

        defaults = dict(rho=rho)
        super(SAM, self).__init__(params, defaults)

        self.base_optimizer = base_optimizer(self.param_groups, lr)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    def _grad_norm(self):
        # Note that p.grad gets the gradient; p.data gets the weight.
        norm = torch.norm(
                    torch.stack([
                        p.grad.norm(p=2)
                        for group in self.param_groups for p in group["params"]
                        if p.grad is not None
                    ]),
                    p=2
               )
        norm += 1e-12 # Avoid zero norm
        return norm

    @torch.no_grad()
    def first_step(self):
        # "Ascent" step: w_t -> w_t + epsilon
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / grad_norm
            for p in group["params"]:
                if p.grad is None:
                    continue
                # Save current parameter for later reversion
                self.state[p]["old_p"] = p.data.clone()
                # Add epsilon in the direction of the gradient
                p.add_(p.grad, alpha=scale)

        # After adding the perturbation, we zero out the gradients
        self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        # Revert parameters to the original values and do a normal optimizer step
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                # Revert to old parameters
                old_p = self.state[p]["old_p"]
                p.data = old_p

        # Now take the "real" step w_{t+1} = w_t - eta * grad(L(w_t + epsilon))
        self.base_optimizer.step()

        # Optionally zero out gradients
        if zero_grad:
            self.zero_grad()

Define an optimizer of `SAM` for the model. We recommend using `SGD` as base optimizer with a learning rate of $0.05$ (which is same with SGD).

In [50]:
base_optimizer = torch.optim.SGD
sam_optimizer = SAM(sparse_model_sam.parameters(), base_optimizer, lr=0.05)

**(5)** (4 points) Please define the optimizing process of SAM.

In [51]:
import torch.nn.functional as F

def cross_entropy(pred, target):
    return F.cross_entropy(pred, target)

def enable_running_stats(model):
    # You might want to enable batchnorm tracking here if using BN
    model.train()

def disable_running_stats(model):
    # You might want to freeze/disable BN tracking here
    model.train()

def optimize_sam(model, optimizer, img, label):

    # --- FIRST FORWARD/BACKWARD PASS (with BN stats enabled) ---
    enable_running_stats(model)

    # 1) Forward
    outputs = model(img)
    loss = cross_entropy(outputs, label)

    # 2) Backward
    loss.backward()

    # 3) First SAM step (adds epsilon to weights)
    optimizer.first_step()

    # --- SECOND FORWARD/BACKWARD PASS (with BN stats disabled) ---
    disable_running_stats(model)

    # 1) Forward again on w + epsilon
    outputs = model(img)
    loss = cross_entropy(outputs, label)

    # 2) Backward
    loss.backward()

    # 3) Second SAM step (revert to original w, then do normal SGD step)
    optimizer.second_step()


In [52]:
train_model(sparse_model_sam, sam_optimizer, optimize_sam, finetune_epoch)

Epoch 0 with 0.96 accuracy on the validation set.
Epoch 1 with 0.964 accuracy on the validation set.
Epoch 2 with 0.965 accuracy on the validation set.
Epoch 3 with 0.966 accuracy on the validation set.
Epoch 4 with 0.969 accuracy on the validation set.
Epoch 5 with 0.972 accuracy on the validation set.
Epoch 6 with 0.97 accuracy on the validation set.
Epoch 7 with 0.976 accuracy on the validation set.
Epoch 8 with 0.974 accuracy on the validation set.
Epoch 9 with 0.976 accuracy on the validation set.


#### Evaluate model
Evaluate the sparse model finetuned with SAM on the test set.

In [53]:
acc = evaluate(sparse_model_sam)
print(f"Accuracy of {round(acc, 4)} on the test set.")

Accuracy of 0.9791 on the test set.


**(6)** (2 points) Give a conclusion comparing SAM with SGD. Is there any drawback of SAM?

### Conclusion: SAM vs. SGD

- **Generalization**: SAM improves generalization by minimizing loss in a smoother, more robust neighborhood, while SGD only minimizes the loss at the current point.
- **Robustness**: SAM is more robust to noisy data and harder-to-fit examples.
- **Drawbacks of SAM**: 
   - Higher computational cost (requires two forward-backward passes).
   - More complex to implement and tune.
   - Potentially unnecessary overhead in cases where the loss landscape is smooth.

Overall, SAM enhances performance but at the cost of efficiency and complexity.
